# Train LSTM on Colab

This notebook sets up Colab, copies `data/lstm_processed/lstm_v1` to local disk, and launches the reusable source training script `src/training/lstm_training/train_lstm.py`.

## 1. Mount Drive

In [2]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Paths

In [3]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for large data/checkpoints/logs.
# Set this to your GitHub repo URL before running the cell.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")
DATASET_NAME = "lstm_v1"

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
DRIVE_DATA_DIR = DRIVE_ROOT / "data" / "lstm_processed" / DATASET_NAME
LOCAL_DATA_DIR = Path("/content/lstm_processed") / DATASET_NAME
DATA_DIR = LOCAL_DATA_DIR

CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "lstm_v1"
MODEL_DIR = DRIVE_ROOT / "models" / "lstm_v1"
LOG_DIR = DRIVE_ROOT / "logs" / "lstm_v1"

os.chdir(PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Current working directory:", Path.cwd())

Code directory: /content/Seoul_bike_project
Drive artifact root: /content/drive/MyDrive/Seoul_bike_project
Drive dataset: /content/drive/MyDrive/Seoul_bike_project/data/lstm_processed/lstm_v1
Local dataset: /content/lstm_processed/lstm_v1
Current working directory: /content/Seoul_bike_project


## 3. Install Requirements

In [4]:
%cd /content/Seoul_bike_project
!python -m pip install -r requirements.txt

/content


## 4. Copy Dataset to Colab Disk

In [5]:
import shutil

RELOAD_LOCAL_DATA = True
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f"Missing dataset directory: {DRIVE_DATA_DIR}")
if LOCAL_DATA_DIR.exists() and RELOAD_LOCAL_DATA:
    shutil.rmtree(LOCAL_DATA_DIR)
if not LOCAL_DATA_DIR.exists():
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
print("Using local dataset:", DATA_DIR)

Using local dataset: /content/lstm_processed/lstm_v1


## 5. Verify Dataset

In [6]:
import json

required = [
    DATA_DIR / "dynamic_features.npy",
    DATA_DIR / "targets.npy",
    DATA_DIR / "targets_raw.npy",
    DATA_DIR / "static_numeric.npy",
    DATA_DIR / "sample_index_train.npy",
    DATA_DIR / "sample_index_val.npy",
    DATA_DIR / "feature_config.json",
    DATA_DIR / "dataset_summary.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset files:\n" + "\n".join(missing))
with open(DATA_DIR / "dataset_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print(json.dumps({
    "T_total": summary["T_total"],
    "S": summary["S"],
    "samples_per_split": summary["samples_per_split"],
}, indent=2))

{
  "T_total": 21888,
  "S": 2780,
  "samples_per_split": {
    "train": 36406880,
    "val": 4136640,
    "test_2025_winter": 8139840,
    "test_2024_april_june": 12120800
  }
}


## 6. W&B Login

In [ ]:
import os
from getpass import getpass
import wandb

wandb_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata
    wandb_key = wandb_key or userdata.get("WANDB_API_KEY")
except Exception as exc:
    print("Colab Secrets not available from this kernel:", type(exc).__name__)

if not wandb_key:
    wandb_key = getpass("Paste W&B API key: ")

os.environ["WANDB_API_KEY"] = wandb_key.strip()
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

## 7. Train With Source Script

In [ ]:
!python -m src.training.lstm_training.train_lstm \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_processed/lstm_v1 \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_v1 \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_v1 \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_v1 \
  --batch_size 32768 \
  --resume auto \
  --resume_mode auto \
  --wandb_enabled true \
  --smoke_test true